In [1]:
import optuna
from optuna.storages.journal import JournalStorage, JournalFileBackend
import pandas as pd
import os

# === CONFIGURATION ===
MODEL_TYPE = "DeepCNNLSTM"  

DB_DIR = "C:\\Users\\kdmen\\Repos\\pers-gest-cls\\dataset\\optuna_dbs"
FILE_NAME = f"1s3w_pretrain_MOE_collapse_HPO_DeepCNNLSTM"
JOURNAL_PATH = os.path.join(DB_DIR, f"{FILE_NAME}.log")

print(f"Loading study: {FILE_NAME}")
print(f"From path: {JOURNAL_PATH}")

Loading study: 1s3w_pretrain_MOE_collapse_HPO_DeepCNNLSTM
From path: C:\Users\kdmen\Repos\pers-gest-cls\dataset\optuna_dbs\1s3w_pretrain_MOE_collapse_HPO_DeepCNNLSTM.log


In [2]:
# 2. Initialize storage
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# 3. Use the TOP-LEVEL optuna function to get summaries
summaries = optuna.get_all_study_summaries(storage)

if not summaries:
    print("No studies found in this file.")
else:
    for s in summaries:
        print(f"Internal Study Name: '{s.study_name}'")

    STUDY_NAME = s.study_name

Internal Study Name: 'pretrain_hpo_DeepCNNLSTM'


In [3]:
# Initialize the storage backend
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# Load the study
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=storage
)

print(f"Study loaded with {len(study.trials)} trials.")
print(f"Best value (Accuracy): {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Study loaded with 100 trials.
Best value (Accuracy): 0.3500
Best Hyperparameters:
  learning_rate: 0.0004946077115444924
  weight_decay: 0.005025359463739119
  dropout: 0.05
  label_smooth: 0.0
  batch_size: 64
  optimizer: adam
  cnn_base_filters: 64
  lstm_hidden: 64
  bidirectional: False
  head_type: linear


In [ ]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate


In [ ]:
# 1. Plot optimization history
fig_hist = plot_optimization_history(study)
fig_hist.show()


In [ ]:
# 2. Plot Hyperparameter Importance
try:
    fig_imp = plot_param_importances(study)
    fig_imp.show()
except Exception as e:
    print(f"Could not plot importance (might need more trials): {e}")

In [ ]:
# 3. Parallel Coordinate Plot (Visualizes high-dimensional relationships)
fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show()

In [ ]:
# FULL
params_to_plot = ["best_or_last_pretr", "context_emb_dim", "context_pool_type", "episodes_per_epoch_train", "finetuning_approach", "groupnorm_num_groups", 
                  "maml_inner_steps", "maml_inner_steps_eval", "maml_msl_num_epochs", "meta_batchsize", "optimizer", "use_GlobalAvgPooling", "use_maml_msl", "wd"]

# PRETRAINING STUFF / MISC
# Apparently finetuning_approach and use_maml_msl only had 1 value? Yes for finetuning currently, but use_maml_msl should have had true and false as well no? ...
params_to_plot_MISC = ["best_or_last_pretr", "finetuning_approach", "use_maml_msl"]

# ARCHITECTURE? --> What does this even mean? How can this vary if we are using the pretrained model....
params_to_plot_ARCH = ["context_emb_dim", "context_pool_type", "groupnorm_num_groups", "use_GlobalAvgPooling"]

# MAML++
params_to_plot_MAMLPP = ["maml_inner_steps", "maml_inner_steps_eval", "maml_msl_num_epochs", "use_maml_msl"]

# LRs MAML++
params_to_plot_LRS = ["maml_alpha_init", "maml_alpha_init_eval", "outer_lr"]

# Learning HPs
params_to_plot_HPS = ["episodes_per_epoch_train", "meta_batchsize", "optimizer", "wd"]

In [ ]:
from optuna.visualization import plot_slice


In [ ]:
fig_slice = plot_slice(study, params=params_to_plot_MISC)
fig_slice.show()

In [ ]:
fig_slice = plot_slice(study, params=params_to_plot_ARCH)
fig_slice.show()

In [ ]:
fig_slice = plot_slice(study, params=params_to_plot_MAMLPP)
fig_slice.show()

In [ ]:
fig_slice = plot_slice(study, params=params_to_plot_LRS)
fig_slice.show()

In [ ]:
fig_slice = plot_slice(study, params=params_to_plot_HPS)
fig_slice.show()

In [ ]:
trials_df = study.trials_dataframe()

# Extract the custom user attributes into the dataframe
trials_df['mean_pretrain_val_acc'] = [t.user_attrs.get('mean_pretrain_val_acc') for t in study.trials]
trials_df['fold_mean_accs'] = [t.user_attrs.get('fold_mean_accs') for t in study.trials]

# Filter for relevant columns and sort by best performance
results_summary = trials_df[['number', 'value', 'mean_pretrain_val_acc', 'datetime_start', 'duration']].sort_values(by='value', ascending=False)
results_summary.head(10)